In [3]:
import scipy.io as sio
from sklearn.decomposition import PCA
from sklearn.metrics import precision_score,f1_score
from torch_geometric.data import InMemoryDataset, Data
import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy.sparse as sp
import numpy as np
import random
import warnings
warnings.filterwarnings('ignore')
from modelUtils import *
class CitationDomainData(InMemoryDataset):
    """
    引文数据集加载器：acmv9,citationv1,dblpv7 originally from CDNE
    """
    def __init__(self,root,name,use_pca=True,pca_dim=1000,transform=None,pre_transform=None,pre_filter=None):
        self.name=name
        self.use_pca = use_pca
        self.dim = pca_dim
        super(CitationDomainData, self).__init__(root, transform, pre_transform, pre_filter)
        self.data, self.slices = torch.load(self.processed_paths[0])
    @property
    def raw_file_names(self):
        return [f'{self.name}.mat']
    @property
    def processed_file_names(self):
        if self.use_pca:
            return [f'data_pca_{self.dim}.pt']
        else:
            return ['data.pt']

    def feature_compression(self,features):
        """Preprcessing of features"""
        features = features.toarray()
        feat = sp.lil_matrix(PCA(n_components=self.dim, random_state=0).fit_transform(features))
        return feat.toarray()

    def process(self):
        net = sio.loadmat(self.raw_dir+"\\"+self.name+".mat")
        features, adj, labels = net['attrb'], net['network'], net['group']
        if not isinstance(features, sp.lil_matrix):
            features = sp.lil_matrix(features)
        # citation networks do not use PCA, but blog networks use
        if self.use_pca:
            features = self.feature_compression(features)
            features = torch.from_numpy(features).to(torch.float)
        else:
            features = torch.from_numpy(features.todense()).to(torch.float)
        if not isinstance(adj,sp.coo_matrix):
            adj = sp.coo_matrix(adj)
        # label is float: to support BCEWithLogits loss
        y = torch.from_numpy(np.array(labels)).to(torch.float)
        data_list = []
        graph = Data(x=features,edge_index=adj,y=y)
        # train-val-test split
        random_node_indices = np.random.permutation(y.shape[0])
        train_size = int(len(random_node_indices) * 0.7)
        val_size = int(len(random_node_indices) * 0.1)
        train_node_indices = random_node_indices[:train_size]
        val_node_indices = random_node_indices[train_size:train_size+val_size]
        test_node_indices = random_node_indices[train_size+val_size:]
        train_mask = torch.zeros([y.shape[0]], dtype=torch.uint8)
        train_mask[train_node_indices] = 1
        val_mask = torch.zeros([y.shape[0]], dtype=torch.uint8)
        val_mask[val_node_indices] = 1
        test_mask = torch.zeros([y.shape[0]], dtype=torch.uint8)
        test_mask[test_node_indices] = 1
        graph.train_mask = train_mask
        graph.val_mask = val_mask
        graph.test_mask = test_mask
        if self.pre_transform is not None:
            graph = self.pre_transform(graph)
        data_list.append(graph)
        data, slices = self.collate(data_list)
        torch.save((data, slices), self.processed_paths[0])

class FE(nn.Module):
    def __init__(self,input_dim,output_dim):
        self.input_dim = input_dim
        self.output_dim = output_dim
        super(FE,self).__init__()
        self.h1 = 512
        self.fc1 = nn.Linear(input_dim,self.h1)
        self.bn1 = nn.BatchNorm1d(self.h1)
        self.fc2 = nn.Linear(self.h1,self.output_dim)
        self.bn2 = nn.BatchNorm1d(self.output_dim)

    def forward(self,inp):
        feat1 =  F.dropout(F.relu(self.bn1(self.fc1(inp))))  # 128-dim feature
        feat2 = F.relu(self.bn2(self.fc2(feat1)))
        return feat1,feat2

class Classifier(nn.Module):
    def __init__(self,input_dim,output_dim):
        self.input_dim = input_dim
        self.output_dim = output_dim
        super(Classifier, self).__init__()
        self.fc3 = nn.Linear(self.input_dim, self.output_dim)

    def forward(self, inp):
        logits = self.fc3(inp)
        return logits

def weight_init(m):
    classname = m.__class__.__name__
    if classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0.1)
    elif classname.find('Linear') != -1:
        nn.init.kaiming_normal_(m.weight.data)
        if m.bias.data is not None:
            torch.nn.init.constant_(m.bias.data, val=0.1)

def f1_scores(y_pred, y_true):
    """ y_pred: prob  y_true 0/1 """
    def predict(y_tru, y_pre):
        top_k_list = y_tru.sum(1)
        prediction = []
        for i in range(y_tru.shape[0]):
            pred_i = torch.zeros(y_tru.shape[1])
            pred_i[torch.argsort(y_pre[i, :])[-int(top_k_list[i].item()):]] = 1
            prediction.append(torch.reshape(pred_i, (1, -1)))
        prediction = torch.cat(prediction, dim=0)
        return prediction.to(torch.int32)
    results = {}
    predictions = predict(y_true, y_pred)
    averages = ["micro", "macro"]
    for average in averages:
        results[average] = f1_score(y_true.cpu().numpy(), predictions.cpu().numpy(), average=average)
    return results


def test(tgt_inp,net,clf):
    net.eval()
    with torch.no_grad():
        test_inp = tgt_inp.x
        test_label = tgt_inp.y
        _,test_feat2 = net(test_inp)
        test_out = clf(test_feat2)
    pred = F.sigmoid(test_out)
    result = f1_scores(pred,test_label)
    return result



In [4]:
def train_test_loop(src_inp,tgt_inp,net,clf,criterion,batch_size,init_lr):
    # clear model before each training
    net.zero_grad()
    # use all labeled source data to train
    src_train_data = src_inp.x
    src_train_label = src_inp.y
    src_train_num = src_train_data.shape[0]
    tgt_train_data = tgt_inp.x
    tgt_train_num = tgt_train_data.shape[0]
    time = 0
    running_clf_loss = 0.
    running_mmd_loss = 0.
    minibatch_times = int(src_train_num/batch_size)+1
    for epoch in range(epoches):
        # mini-batch
        for start in range(0,src_train_num,batch_size):
            net.train()
            end = min(src_train_num, start+batch_size)
            src_input = src_train_data[start:end,:]
            src_label = src_train_label[start:end,:]
            src_feat1, src_feat2 = net(src_input)
            src_output = clf(src_feat2)
            clf_loss = criterion['clf'](src_output,src_label)
            # 如果此时target domain还有至少完整的一个batch数据，则利用它计算MMD,注意source target的minibatch维度必须相同
            mmd_loss = torch.tensor(0.)
            if start<tgt_train_num-batch_size:
                end_ = min(tgt_train_num, start+batch_size)
                tgt_input = tgt_train_data[start:end_,:]
                tgt_feat1,tgt_feat2 = net(tgt_input)
                mmd_loss = 0.5*(criterion['mmd'](src_feat1,tgt_feat1)+criterion['mmd'](src_feat2,tgt_feat2))
            lr = init_lr/(1+10*epoch)**0.75
            #optimizer = torch.optim.SGD(net.parameters(),lr, 0.9,weight_decay=1e-3/2)
            optimizer = torch.optim.Adam(net.parameters(),lr,weight_decay=5e-3/2)
            optimizer.zero_grad()
            loss = clf_loss+mmd_loss
            loss.backward()
            optimizer.step()
            time += 1
            running_clf_loss += clf_loss.item()
            running_mmd_loss += mmd_loss.item()
            if time%50==0 and time>0:
                train_result = test(src_inp,net,clf)
                test_result = test(tgt_inp,net,clf)
                print(f"CLF Loss:{running_clf_loss/50}, MMD Loss:{running_mmd_loss/50},Step:{epoch*minibatch_times+time}, TrainMacroF1:{train_result['macro']}, TrainMicroF1:{train_result['micro']},TestMacroF1:{test_result['macro']}, TestMicroF1:{test_result['micro']}")
                running_clf_loss = 0.
                running_mmd_loss = 0.
random.seed(200)
np.random.seed(200)
torch.manual_seed(200)
src = CitationDomainData("data/acmv9",name="acmv9",use_pca=False)
tgt = CitationDomainData("data/citationv1",name="citationv1",use_pca=False)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
epoches = 30
lr = 0.03
batch_size = 100
clf_criterion = nn.BCEWithLogitsLoss()
sigmas = [
    1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1, 5, 10, 15, 20, 25, 30, 35, 100,
    1e3, 1e4, 1e5, 1e6
]
kernels = []
for sigma in sigmas:
    kernels.append(GaussianKernel(sigma=sigma))
mmd_criterion = MultipleKernelMaximumMeanDiscrepancy(kernels=kernels)
criterion = {"clf":clf_criterion,"mmd":mmd_criterion}
source_data = src[0]
source_data = source_data.to(device)
target_data = tgt[0]
target_data = target_data.to(device)
print(source_data.x.shape[0],target_data.x.shape[0])
#net = FE(input_dim=source_data.x.shape[1],output_dim=128)
#net.apply(weight_init)
#net = net.to(device)
#clf = Classifier(input_dim=128,output_dim=source_data.y.shape[1])
#clf.apply(weight_init)
#clf = clf.to(device)
#train_test_loop(source_data, target_data, net, clf, criterion,batch_size,init_lr=lr)

cuda
9360 8935


In [8]:
from modelUtils import MMD_loss,GaussianKernel,MultipleKernelMaximumMeanDiscrepancy
import matplotlib.pyplot as plt
import torch
datasets = ["acmv9","citationv1","dblpv7"]
for i in range(len(datasets)):
    for j in range(i+1,len(datasets)):
        dual_ds = [(i,j),(j,i)]
        for ds in dual_ds:
            src_name = datasets[ds[0]]
            tgt_name = datasets[ds[1]]
            print(src_name, tgt_name)
            src = CitationDomainData(f"data/{src_name}",name=src_name,use_pca=False)
            tgt = CitationDomainData(f"data/{tgt_name}",name=tgt_name,use_pca=False)
            source_data = src[0]
            source_data = source_data.to(device)
            target_data = tgt[0]
            target_data = target_data.to(device)
            size = min(source_data.x.shape[0],target_data.x.shape[0])
            random_indices = np.random.permutation(size)
            x = source_data.x[random_indices]
            y = target_data.x[random_indices]
            sigmas = [
                1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1, 5, 10, 15, 20, 25, 30, 35, 100,
                1e3, 1e4, 1e5, 1e6
            ]
            kernels = []
            for sigma in sigmas:
                kernels.append(GaussianKernel(sigma=sigma))
            attr_mmd = 0.
            mmd2 = MultipleKernelMaximumMeanDiscrepancy(kernels=kernels)
            bs = 64
            for start in range(0,size,bs):
                end = min(size-1,start+bs)
                attr_mmd += mmd2(x[start:end],y[start:end])
            print(src_name, tgt_name, attr_mmd)

acmv9 citationv1
acmv9 citationv1 tensor(6.4770, device='cuda:0')
citationv1 acmv9
citationv1 acmv9 tensor(6.5710, device='cuda:0')
acmv9 dblpv7
acmv9 dblpv7 tensor(6.4672, device='cuda:0')
dblpv7 acmv9
dblpv7 acmv9 tensor(6.3061, device='cuda:0')
citationv1 dblpv7
citationv1 dblpv7 tensor(5.3429, device='cuda:0')
dblpv7 citationv1
dblpv7 citationv1 tensor(5.3840, device='cuda:0')
